# 📋 Schema Setup and Verification
## Unity Catalog External Schemas (AWS S3)

**Purpose**: Verify that external schemas exist and are accessible

**External Schemas** (S3-backed via Unity Catalog):
- 🥉 **Bronze**: `workspace.`workspace-adventureworks-bronze``
- 🥈 **Silver**: `workspace.`workspace-adventureworks-silver``
- 🥇 **Gold**: `workspace.`workspace-adventureworks-gold``

**Storage**: AWS S3 external locations configured in Unity Catalog

**Note**: Schemas should be created manually with appropriate S3 locations and IAM permissions before running this notebook.

In [0]:
from pyspark.sql import SparkSession
from datetime import datetime

# Schema configuration (S3 External Storage)
BRONZE_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"
GOLD_SCHEMA = "workspace.`workspace-adventureworks-gold`"

print("=" * 80)
print("📋 SCHEMA SETUP AND VERIFICATION")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Storage: AWS S3 External (Unity Catalog)")
print("=" * 80)
print()

In [0]:
def verify_schema(catalog_name, schema_name, display_name):
    """
    Verify that a schema exists and display its properties
    """
    try:
        # Check if schema exists
        schema_full = f"{catalog_name}.`{schema_name}`"
        schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()
        schema_names = [row.databaseName for row in schemas]
        
        if schema_name in schema_names:
            # Get schema details
            details = spark.sql(f"DESCRIBE SCHEMA EXTENDED {schema_full}").collect()
            details_dict = {row[0]: row[1] for row in details}
            
            location = details_dict.get('Location', 'N/A')
            comment = details_dict.get('Comment', 'N/A')
            
            print(f"✅ {display_name}")
            print(f"   Schema: {schema_full}")
            print(f"   Location: {location}")
            print(f"   Comment: {comment}")
            print(f"   Status: ✓ EXISTS")
            
            # Verify it's external (S3)
            if location.startswith('s3://'):
                print(f"   Storage Type: ✓ S3 EXTERNAL")
            else:
                print(f"   ⚠️  Warning: Location does not appear to be S3: {location}")
            
            print()
            return True
        else:
            print(f"❌ {display_name}")
            print(f"   Schema: {schema_full}")
            print(f"   Status: NOT FOUND")
            print(f"   Action Required: Create schema with S3 external location")
            print()
            return False
            
    except Exception as e:
        print(f"❌ {display_name}")
        print(f"   Error: {str(e)}")
        print()
        return False

# Verify all schemas
print("\n🔍 VERIFYING EXTERNAL SCHEMAS...\n")

bronze_ok = verify_schema("workspace", "workspace-adventureworks-bronze", "🥉 Bronze Schema")
silver_ok = verify_schema("workspace", "workspace-adventureworks-silver", "🥈 Silver Schema")
gold_ok = verify_schema("workspace", "workspace-adventureworks-gold", "🥇 Gold Schema")

print("=" * 80)
if bronze_ok and silver_ok and gold_ok:
    print("✅ ALL SCHEMAS VERIFIED SUCCESSFULLY")
    print("=" * 80)
    print("\n✓ Ready to proceed with data pipeline")
else:
    print("⛔ SCHEMA VERIFICATION FAILED")
    print("=" * 80)
    print("\n❌ One or more schemas not found or not accessible")
    print("\nPlease create missing schemas using:")
    print("\n-- Bronze Schema")
    print("CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-bronze`")
    print("LOCATION 's3://your-bucket/adventureworks/bronze'")
    print("COMMENT 'Bronze layer - Raw data with audit columns';")
    print("\n-- Silver Schema")
    print("CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-silver`")
    print("LOCATION 's3://your-bucket/adventureworks/silver'")
    print("COMMENT 'Silver layer - Cleaned and conformed data';")
    print("\n-- Gold Schema")
    print("CREATE SCHEMA IF NOT EXISTS workspace.`workspace-adventureworks-gold`")
    print("LOCATION 's3://your-bucket/adventureworks/gold'")
    print("COMMENT 'Gold layer - Business aggregates and analytics';")
    raise Exception("Schema verification failed. Cannot proceed.")

In [0]:
# List existing tables in each schema (if any)
print("\n" + "=" * 80)
print("📊 EXISTING TABLES IN SCHEMAS")
print("=" * 80)

for schema, name in [(BRONZE_SCHEMA, "Bronze"), (SILVER_SCHEMA, "Silver"), (GOLD_SCHEMA, "Gold")]:
    try:
        tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
        if tables:
            print(f"\n{name} Schema ({schema}):")
            for table in tables:
                print(f"  - {table.tableName}")
        else:
            print(f"\n{name} Schema ({schema}): No tables yet (empty)")
    except Exception as e:
        print(f"\n{name} Schema: Error listing tables - {str(e)}")

print("\n" + "=" * 80)

In [0]:
# Return success for orchestration workflow
print("\n✅ Schema setup verification completed successfully")
print(f"Completion Time: {datetime.now()}")

dbutils.notebook.exit("SUCCESS")